# Sequence Labeling: Viterbi from scratch, the CRF that beats greedy, and the metric that lies

You knew, reading *"Jane Smith flew to New York City,"* that *Smith* was a surname and *New York City* a place — and you used the **neighbours** to know it. **Sequence labeling** teaches a model to assign a tag to **every token** where each tag depends on the tags around it: POS tagging, NER, chunking, slot filling. This notebook builds the structured-prediction core from scratch and proves three things you can *see*:

1. **Viterbi beats greedy** — the locally-best tag at each step is *not* the globally-best sequence.
2. **A globally-scored sequence rules out illegal transitions** — `O → I-PER` is impossible, even when the per-token emission for `I-PER` is highest.
3. **Token accuracy lies; entity F1 is honest** — one dropped token of a span halves F1 while accuracy barely moves.

Every function comes from `sequence_labeling.py` (the single source of truth this notebook, the page, and the figures all share). Each code cell **asserts its point, then prints**.

## Setup — import the seeded backend

The structured-prediction math here is **pure-Python / numpy**, so the device is honestly CPU. The module seeds numpy once at import (deterministic runs) and uses a **stable md5 hash**, never Python's per-process-salted `hash()`. We print an honest runtime banner.

In [1]:
from sequence_labeling import runtime_banner

print(runtime_banner())

device: cpu (pure-Python/numpy) | python 3.12.13 | numpy 2.4.6, torch 2.12.0 (not used here)


## The HMM: a generative story for a tagged sentence

A Hidden Markov Model walks through hidden states (the tags) and emits a word at each: pick a first tag from $\pi$, emit a word from that tag's vocabulary distribution $b$, transition to the next tag with $a$, repeat. Two Markov assumptions make it tractable — the next tag depends only on the current tag, and a word depends only on its own tag — giving the joint factorization
$$p(y_{1:n}, x_{1:n}) = \pi(y_1)\,b_{y_1}(x_1)\prod_{i=2}^{n} a_{y_{i-1},y_i}\,b_{y_i}(x_i).$$
We load the tiny 2-tag (Noun/Verb) HMM used throughout the chapter.

In [2]:
from sequence_labeling import HMM_TAGS, HMM_START, HMM_TRANS, HMM_EMIT

# the parameters of the 'time flies fast' HMM (every word can be Noun OR Verb)
assert HMM_TAGS == ('N', 'V')
assert abs(sum(HMM_START.values()) - 1.0) < 1e-9          # pi is a distribution
assert abs(HMM_TRANS[('N','N')] + HMM_TRANS[('N','V')] - 1.0) < 1e-9   # rows of a sum to 1
print('tags      :', HMM_TAGS)
print('start  pi :', HMM_START)
print('trans   a :', HMM_TRANS)
print('emit    b :', HMM_EMIT)

tags      : ('N', 'V')
start  pi : {'N': 0.5, 'V': 0.5}
trans   a : {('N', 'N'): 0.6, ('N', 'V'): 0.4, ('V', 'N'): 0.8, ('V', 'V'): 0.2}
emit    b : {('N', 'time'): 0.6, ('V', 'time'): 0.3, ('N', 'flies'): 0.3, ('V', 'flies'): 0.7, ('N', 'fast'): 0.2, ('V', 'fast'): 0.7}


## Decoding 1 — Viterbi: the most probable tag *path*

Decoding asks for $\arg\max_y p(y \mid x)$ over the $k^n$ possible tag sequences — without enumerating them. **Viterbi** is the dynamic program that does it in $O(nk^2)$ by keeping, at each state, the single best path *into* that state:
$$\delta_t(j) = \Big[\max_i \delta_{t-1}(i)\,a_{ij}\Big]\,b_j(x_t),\qquad \delta_1(j)=\pi_j\,b_j(x_1),$$
with a backpointer per cell. We run it on the genuinely ambiguous *"time flies fast."*

In [3]:
from sequence_labeling import viterbi

words = ['time', 'flies', 'fast']
v_path, v_prob, delta = viterbi(words)

# delta[t, j] is the best-path probability into tag j at position t (the Viterbi variable)
assert v_path == ['N', 'N', 'V'], 'Viterbi reads time/flies as a noun phrase, fast as the verb'
assert delta.shape == (3, 2)
print('Viterbi path :', v_path)
print('path prob    : %.5f' % v_prob)
print('delta table  (rows = t, cols = N,V):')
print(delta)

Viterbi path : ['N', 'N', 'V']
path prob    : 0.01512
delta table  (rows = t, cols = N,V):
[[0.3     0.15   ]
 [0.054   0.084  ]
 [0.01344 0.01512]]


## Decoding 2 — greedy per-token argmax (the baseline Viterbi must beat)

A *greedy* decoder walks left to right and at every position commits to the locally-best tag, never reconsidering. That early commitment is a trap whenever a tempting local emission leads into a worse continuation. On *"time flies fast"* the high local verb emission $b_V(\text{flies})=0.7$ seduces greedy into committing to V at *flies* — and locks it into a globally worse path.

In [4]:
from sequence_labeling import greedy_argmax, path_probability

g_path = greedy_argmax(words)
v_score = path_probability(words, v_path)
g_score = path_probability(words, g_path)

# the whole lesson: greedy DISAGREES with Viterbi and scores strictly lower
assert g_path == ['N', 'V', 'N'], 'greedy is seduced by b_V(flies)=0.7'
assert g_path != v_path
assert v_score > g_score, 'Viterbi finds the true global best; greedy lands on 2nd-best'
print('greedy  path : %s   p(y,x) = %.5f   <- locally tempting, globally 2nd-best' % (g_path, g_score))
print('Viterbi path : %s   p(y,x) = %.5f   <- the true global best' % (v_path, v_score))
print('Viterbi / greedy probability ratio: %.3f' % (v_score / g_score))

greedy  path : ['N', 'V', 'N']   p(y,x) = 0.01344   <- locally tempting, globally 2nd-best
Viterbi path : ['N', 'N', 'V']   p(y,x) = 0.01512   <- the true global best
Viterbi / greedy probability ratio: 1.125


## The other semiring — forward (sum) computes the partition function

The **forward** algorithm uses the *same* trellis with $\max$ replaced by $\sum$: $\alpha_t(j)=[\sum_i \alpha_{t-1}(i)\,a_{ij}]\,b_j(x_t)$, giving the *total* probability $p(x)=\sum_j \alpha_n(j)$ — which you need for **training** and for a CRF's partition function $Z(x)$, **not** the single best path. Remember: **Viterbi = max, forward = sum.** The total over all paths must be $\ge$ the single best path's probability.

In [5]:
import numpy as np
from sequence_labeling import forward_logZ

log_px = forward_logZ(words)
px = np.exp(log_px)

# the sum over ALL paths must be >= the single best path (Viterbi), and <= 1
assert px >= v_prob - 1e-12, 'total p(x) cannot be less than the best single path'
assert 0.0 < px <= 1.0
print('forward  log p(x) = %.5f   ->  p(x) = %.5f  (sum over ALL paths)' % (log_px, px))
print('Viterbi  best path p = %.5f  (single best path)' % v_prob)
print('p(x) >= best path :', px >= v_prob)

forward  log p(x) = -2.69563   ->  p(x) = 0.06750  (sum over ALL paths)
Viterbi  best path p = 0.01512  (single best path)
p(x) >= best path : True


## NER labels and the BIO grammar — encode the illegal transition

Switching to NER: a 5-tag label set `O, B-PER, I-PER, B-LOC, I-LOC`. A *linear-chain CRF* scores a whole sequence as **emissions + transitions**, and a trained transition matrix encodes the BIO grammar. The key structural fact: `I-TYPE` is legal **only** after `B-TYPE`/`I-TYPE` of the *same* type — so `O → I-PER` gets a score of $-\infty$. We build the matrix and check that constraint.

In [6]:
from sequence_labeling import NER_TAGS, TAG_INDEX, bio_transition_matrix, NEG_INF

A = bio_transition_matrix()
o, b_per, i_per, b_loc, i_loc = (TAG_INDEX[t] for t in NER_TAGS)

# the illegal transition O -> I-PER is -inf; the legal continuation B-PER -> I-PER is not
assert A[o, i_per] == NEG_INF, 'O -> I-PER must be impossible'
assert A[b_loc, i_per] == NEG_INF, 'B-LOC -> I-PER must be impossible (wrong type)'
assert A[b_per, i_per] > 0, 'B-PER -> I-PER is the legal continuation'
print('tags:', NER_TAGS)
print('A[O      -> I-PER] = %.0e   <- illegal' % A[o, i_per])
print('A[B-LOC  -> I-PER] = %.0e   <- illegal (wrong type)' % A[b_loc, i_per])
print('A[B-PER  -> I-PER] = %+.1f      <- legal continuation' % A[b_per, i_per])

tags: ('O', 'B-PER', 'I-PER', 'B-LOC', 'I-LOC')
A[O      -> I-PER] = -1e+09   <- illegal
A[B-LOC  -> I-PER] = -1e+09   <- illegal (wrong type)
A[B-PER  -> I-PER] = +1.5      <- legal continuation


## CRF decoding rules out the illegal transition

Now the payoff. We give the decoder emissions that **wrongly** top out on `I-PER` at token 1, even though token 0 is `O`. A per-token argmax falls for it and emits the illegal `O → I-PER`. The CRF, scoring the whole sequence with the $-\infty$ transition, **rules that path out** and backs off to the best *legal* tagging — the correct `O, B-LOC`.

In [7]:
from sequence_labeling import crf_viterbi_decode, crf_sequence_score

emissions = np.array([
    # O     B-PER  I-PER  B-LOC  I-LOC
    [2.0,   0.1,   0.1,   0.5,   0.1],   # token 0 'downtown' -> clearly O
    [0.5,   0.4,   2.2,   1.9,   0.2],   # token 1 'Paris'    -> emission WRONGLY tops I-PER
])
greedy_tags = [NER_TAGS[int(np.argmax(emissions[t]))] for t in range(2)]
crf_tags, crf_best = crf_viterbi_decode(emissions, A)
s_illegal = crf_sequence_score(emissions, ['O', 'I-PER'], A)

assert greedy_tags == ['O', 'I-PER'], 'per-token argmax emits the ILLEGAL I-PER'
assert crf_tags == ['O', 'B-LOC'], 'CRF rules out O->I-PER, recovers the correct O,B-LOC'
assert crf_best > s_illegal
print('per-token argmax :', greedy_tags, '  <- ILLEGAL (I-PER with no B-PER before it)')
print('CRF Viterbi      :', crf_tags, '  <- the correct, legal reading')
print('score(O, I-PER)  = %.0e   <- O->I-PER = -inf' % s_illegal)
print('score(O, B-LOC)  = %+.1f' % crf_best)

per-token argmax : ['O', 'I-PER']   <- ILLEGAL (I-PER with no B-PER before it)
CRF Viterbi      : ['O', 'B-LOC']   <- the correct, legal reading
score(O, I-PER)  = -1e+09   <- O->I-PER = -inf
score(O, B-LOC)  = +4.9


## CRF sequence scoring keeps a span consistent

Beyond illegal transitions, global scoring keeps spans *consistent*. On a 3-token person name *"Dr Jane Smith"* the emissions slightly favour `B-PER` again on the last token — so an independent per-token argmax **fragments** the name into two spans. The CRF keeps one clean `B-PER, I-PER, I-PER` person span.

In [8]:
em3 = np.array([
    # O    B-PER  I-PER  B-LOC  I-LOC
    [0.2,  2.5,   0.1,   0.3,   0.1],   # 'Dr'
    [0.2,  1.0,   2.2,   0.3,   0.1],   # 'Jane'
    [0.2,  1.5,   1.4,   0.3,   0.1],   # 'Smith' -> emission torn B-PER vs I-PER
])
indep = [NER_TAGS[int(np.argmax(em3[t]))] for t in range(3)]
crf3, _ = crf_viterbi_decode(em3, A)

assert indep == ['B-PER', 'I-PER', 'B-PER'], 'independent argmax breaks the span at Smith'
assert crf3 == ['B-PER', 'I-PER', 'I-PER'], 'CRF keeps one consistent 3-token PER span'
print('independent argmax :', indep, '  <- stray B-PER fragments the name into two spans')
print('CRF Viterbi        :', crf3, '  <- one clean 3-token PERSON span')

independent argmax : ['B-PER', 'I-PER', 'B-PER']   <- stray B-PER fragments the name into two spans
CRF Viterbi        : ['B-PER', 'I-PER', 'I-PER']   <- one clean 3-token PERSON span


## BIO span decoding — turn tags into countable entities

Evaluation needs *entities*, not tags. `decode_bio_spans` recovers `(type, start, end)` tuples: a span opens at each `B-`, extends through matching `I-`, and closes at the next `B-`/`O`/type-change — exactly how CoNLL/seqeval count entities.

In [9]:
from sequence_labeling import decode_bio_spans

tags = ['B-PER', 'I-PER', 'O', 'B-LOC', 'I-LOC', 'I-LOC', 'O', 'O']
spans = decode_bio_spans(tags)

# 'Jane Smith' = PER over tokens 0-1; 'New York City' = LOC over tokens 3-5
assert spans == {('PER', 0, 1), ('LOC', 3, 5)}, 'two spans, recovered by merging B...I runs'
print('tags  :', tags)
print('spans :', sorted(spans))

tags  : ['B-PER', 'I-PER', 'O', 'B-LOC', 'I-LOC', 'I-LOC', 'O', 'O']
spans : [('LOC', 3, 5), ('PER', 0, 1)]


## The metric that lies — token accuracy vs entity F1

**You do not score NER by token accuracy.** A span is all-or-nothing, but token accuracy gives partial credit and is swamped by the easy `O` tokens. Drop one token of a 3-token entity and watch token accuracy barely move while entity F1 *halves*. We compute both from scratch.

In [10]:
from sequence_labeling import token_accuracy, entity_prf

gold = ['B-PER', 'I-PER', 'O', 'B-LOC', 'I-LOC', 'I-LOC', 'O', 'O']   # 'New York City' = LOC
pred = ['B-PER', 'I-PER', 'O', 'B-LOC', 'I-LOC', 'O',     'O', 'O']   # model drops 'City'
acc = token_accuracy(gold, pred)
p, r, f1 = entity_prf(gold, pred)

assert abs(acc - 0.875) < 1e-9, '7 of 8 tokens correct'
assert abs(f1 - 0.5) < 1e-9, 'one dropped token halves entity F1'
print('token accuracy = %.3f   <- looks fine (swamped by easy O tokens)' % acc)
print('entity  P=%.3f  R=%.3f  F1=%.3f   <- one dropped token halves it' % (p, r, f1))

token accuracy = 0.875   <- looks fine (swamped by easy O tokens)
entity  P=0.500  R=0.500  F1=0.500   <- one dropped token halves it


## Cross-check against `seqeval` (the de-facto CoNLL metric)

Our from-scratch `entity_prf` should agree exactly with `seqeval`, the standard Python port of CoNLL's `conlleval`. If `seqeval` isn't installed the cell skips gracefully — the from-scratch numbers above stand on their own.

In [11]:
try:
    from seqeval.metrics import f1_score as seq_f1, precision_score as seq_p, recall_score as seq_r
    sp, sr, sf = seq_p([gold], [pred]), seq_r([gold], [pred]), seq_f1([gold], [pred])
    assert abs(sf - f1) < 1e-9, 'from-scratch entity F1 must match seqeval'
    print('seqeval     P=%.3f  R=%.3f  F1=%.3f' % (sp, sr, sf))
    print('from-scratch P=%.3f  R=%.3f  F1=%.3f' % (p, r, f1))
    print('match:', abs(sf - f1) < 1e-9)
except ImportError:
    print('seqeval not installed — from-scratch entity_prf above is self-contained.')

seqeval     P=0.500  R=0.500  F1=0.500
from-scratch P=0.500  R=0.500  F1=0.500
match: True


## A real pretrained NER model

Finally, a real `dslim/bert-base-NER` pipeline on a sentence — the modern default (a pretrained transformer + a token-classification head). `aggregation_strategy='first'` merges subwords back to word-level spans using the first-subword label (the standard alignment convention). If the model isn't downloadable offline the cell notes that and skips.

In [12]:
try:
    from transformers import pipeline
    ner = pipeline('token-classification', model='dslim/bert-base-NER',
                   aggregation_strategy='first')
    text = 'Jane Smith flew from New York to the Acme Corporation office in Paris.'
    ents = ner(text)
    groups = [e['entity_group'] for e in ents]
    assert 'PER' in groups and 'LOC' in groups and 'ORG' in groups, 'expect PER, LOC, ORG'
    print('NER on:', text)
    for e in ents:
        print('  %-4s %-20s (score=%.3f)' % (e['entity_group'], e['word'], e['score']))
except Exception as exc:  # offline / model unavailable
    print('skipping real NER (model unavailable offline):', type(exc).__name__)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


NER on: Jane Smith flew from New York to the Acme Corporation office in Paris.
  PER  Jane Smith           (score=0.999)
  LOC  New York             (score=1.000)
  ORG  Acme Corporation     (score=0.999)
  LOC  Paris                (score=1.000)


## What we saw

- **Viterbi beat greedy** on *"time flies fast"*: `N N V` (p = 0.01512) vs greedy's `N V N` (p = 0.01344) — the global best vs the locally-tempting 2nd-best.
- **Forward (sum) ≥ Viterbi (max)** on the same trellis — same DP, different semiring; forward gives $p(x)$ / the CRF partition function, Viterbi gives the single best path.
- **Global scoring ruled out `O → I-PER`** even when its per-token emission was highest, and kept a 3-token person span consistent where independent argmax fragmented it.
- **Token accuracy lied** (0.875) while **entity F1 halved** (0.500) on one dropped token — and our from-scratch metric matched `seqeval` exactly.

The through-line: the *decoding* machinery (Viterbi over emission + transition scores) barely changes from the HMM to the biLSTM-CRF; what improves is **where the emissions come from** — counts → hand features → biLSTM → pretrained transformer. See the [concept page](../09-Sequence-Labeling-POS-and-NER.md) for the full ladder and [references](../09-Sequence-Labeling-POS-and-NER.references.md).